In [452]:
!pip install numpy pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [453]:
import numpy as np
import pandas as pd
import random
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [454]:
categories = [
    "Electricity", "Water", "Sanitation", "Roads", "Telecom",
    "Banking", "Insurance", "Pension", "Education", "Healthcare",
    "Transport", "Documents", "Housing", "Employment", "Law & Order",
    "Railways", "Gas", "Internet Services", "Tax", "Other"
]

In [455]:
categories_keywords = {
    "Electricity": [
        "power cut", "electricity outage", "no power supply", "frequent power cuts",
        "low voltage issue", "high voltage fluctuation", "transformer not working",
        "transformer blast", "short circuit", "meter not working",
        "electric pole damage", "street light not working", "bijli nahi aa rahi",
        "light chali gayi", "current issue","bijli problem",
    "electric issue",
    "electric problem",
    "current issue",
    "power problem"
    ],

    "Water": [
        "water leakage", "pipeline leakage", "no water supply", "irregular water supply",
        "dirty water supply", "sewage water mixing", "water pipeline broken",
        "low water pressure", "water tank overflow", "paani nahi aa raha",
        "paani ganda hai", "pipeline phooti hui hai", "paani leakage","water coming from pipe",
    "pipe se paani aa raha hai",
    "pipe burst",
    "water flowing",
    "water overflow"
    ],

    "Sanitation": [
        "garbage not collected", "overflowing dustbin", "dirty streets",
        "waste management issue", "drainage blockage", "sewer overflow",
        "open garbage dumping", "bad smell from garbage",
        "kooda nahi uth raha", "ganda area", "nali jam hai", "kachra"
    ],

    "Roads": [
        "road damage", "potholes on road", "broken road", "road not constructed",
        "poor road condition", "road repair needed", "waterlogging on road",
        "unsafe road", "gaddha hai road pe", "road kharab hai"
    ],

    "Telecom": [
        "network issue", "call drop problem", "poor signal strength",
        "no network coverage", "mobile network down", "sim not working",
        "signal weak", "network nahi aa raha"
    ],

    "Banking": [
        "account blocked", "transaction failed", "money deducted but not received",
        "atm not working", "online banking issue", "payment failed",
        "unauthorized transaction", "bank server down"
    ],

    "Insurance": [
        "claim rejected", "insurance claim delay", "policy issue",
        "insurance not processed", "premium issue", "claim settlement delay"
    ],

    "Pension": [
        "pension delay", "pension not credited", "pension stopped",
        "old age pension issue", "retirement benefits delay",
        "pension nahi mil raha"
    ],

    "Education": [
        "school issue", "college issue", "fees problem",
        "admission issue", "exam result delay", "scholarship not received",
        "teacher not available"
    ],

    "Healthcare": [
        "hospital issue", "doctor not available", "medicine not available",
        "poor hospital service", "long waiting time", "emergency not handled",
        "ambulance delay", "treatment issue"
    ],

    "Transport": [
        "bus delay", "public transport issue", "no buses available",
        "auto overcharging", "traffic congestion", "transport strike",
        "vehicle not available"
    ],

    "Documents": [
        "aadhaar issue", "pan card issue", "document verification problem",
        "passport delay", "duplicate documents issue",
        "kyc not completed"
    ],

    "Housing": [
        "water leakage wall", "seepage issue", "building crack",
        "house damage", "illegal construction", "poor housing condition",
        "ceiling leakage", "wall damage"
    ],

    "Employment": [
        "job issue", "salary not received", "unemployment issue",
        "job fraud", "delayed salary", "termination issue",
        "workplace harassment"
    ],

    "Law & Order": [
        "theft", "robbery", "police complaint not registered",
        "harassment issue", "safety issue", "crime report",
        "illegal activity", "violence issue"
    ],

    "Railways": [
        "train delay", "train cancelled", "railway issue",
        "dirty railway station", "ticket booking issue",
        "seat not available", "train late"
    ],

    "Gas": [
        "gas leakage", "cylinder issue", "gas not delivered",
        "gas pipeline issue", "lpg problem", "gas connection issue"
    ],

    "Internet Services": [
        "internet slow", "wifi not working", "broadband issue",
        "no internet connection", "network problem",
        "frequent disconnection", "slow speed internet",
        "wifi disconnect ho raha hai", "net"
    ],

    "Tax": [
        "tax issue", "refund delay", "wrong tax calculation",
        "income tax problem", "gst issue", "tax not processed"
    ]
}

In [456]:
def generate_complaint():
    # 1️⃣ Decide single vs multi-label (bias towards multi-label)
    if random.random() < 0.6:
        selected_categories = random.sample(list(categories_keywords.keys()), 2)
    else:
        selected_categories = [random.choice(list(categories_keywords.keys()))]

    # 2️⃣ Paraphrase function (FIXED INDENTATION)
    def paraphrase(text):
        variations = [
            text,
            "issue of " + text,
            "problem with " + text,
            text + " happening",
            "facing " + text,
        ]
        return random.choice(variations)

    # 3️⃣ Pick phrases with paraphrasing
    parts = [paraphrase(random.choice(categories_keywords[cat])) for cat in selected_categories]

    # 4️⃣ Natural connectors
    connectors = [" and ", " with ", " along with ", " causing ", " due to "]

    if len(parts) == 2:
        base_issue = parts[0] + random.choice(connectors) + parts[1]
    else:
        base_issue = parts[0]

    # 5️⃣ Templates
    templates = [
        "there is {}",
        "i am facing {}",
        "i have an issue with {}",
        "{} from last few days",
        "problem related to {}",
        "{} ho raha hai",
        "{} ki problem aa rahi hai",
        "bahut time se {}",
        "issue regarding {}",
        "complaint about {}"
    ]

    sentence = random.choice(templates).format(base_issue)

    # 6️⃣ Urgency
    urgency_phrases = [
        "",
        " please fix this urgently",
        " bahut problem ho rahi hai",
        " this is serious issue",
        " kindly resolve soon",
        " jaldi solve karo",
        " no action has been taken yet",
        " i already complained before",
        " this needs immediate attention"
    ]

    sentence += random.choice(urgency_phrases)

    # 7️⃣ Noise
    noise = [
        "",
        " since 2 days",
        " from last week",
        " still not resolved",
        " again facing same issue",
        " pls help",
        " asap",
        " !!!",
        " ???"
    ]

    sentence += " " + random.choice(noise)

    # 8️⃣ Random casing
    if random.random() < 0.3:
        sentence = sentence.upper()
    elif random.random() < 0.3:
        sentence = sentence.capitalize()

    return sentence.strip(), selected_categories

In [457]:
train_data = [generate_complaint() for _ in range(20000)]
test_data = [generate_complaint() for _ in range(5000)]

In [458]:
train_df = pd.DataFrame(train_data, columns=["text", "labels"])
test_df = pd.DataFrame(test_data, columns=["text", "labels"])

train_df.head()

,text,labels
0,issue regarding kachra happening and sim not w...,"[Sanitation, Telecom]"
1,issue regarding problem with refund delay no a...,[Tax]
2,PROBLEM WITH POWER CUT FROM LAST FEW DAYS JALD...,[Electricity]
3,THERE IS POOR ROAD CONDITION HAPPENING CAUSING...,"[Roads, Tax]"
4,bahut time se pension not credited bahut probl...,[Pension]


In [459]:
df_kaggle = pd.read_csv("customer_support_tickets.csv")
df_kaggle.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [460]:
def map_category(text):
    text = text.lower()
    
    if "payment" in text or "transaction" in text:
        return ["Banking"]
    elif "network" in text or "internet" in text:
        return ["Internet Services"]
    elif "refund" in text:
        return ["Banking"]
    else:
        return ["Other"]

In [461]:
df_kaggle["labels"] = df_kaggle["Ticket Description"].apply(map_category)
df_kaggle["clean_text"] = df_kaggle["Ticket Description"].apply(clean_text)

In [462]:
train_df["clean_text"] = train_df["text"].apply(clean_text)

In [463]:
combined_df = pd.concat([
    train_df[["clean_text", "labels"]],
    df_kaggle[["clean_text", "labels"]]
])

In [464]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [465]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["labels"])
y_test = mlb.transform(test_df["labels"])

In [466]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

In [467]:
model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=1000))

In [468]:
y_probs = model.predict_proba(X_test)
y_pred = (y_probs > 0.25).astype(int)

In [469]:
def predict_complaint(text, threshold=0.25):
    text_clean = clean_text(text)
    vec = vectorizer.transform([text_clean])
    
    probs = model.predict_proba(vec)[0]
    
    labels = set([mlb.classes_[i] for i, p in enumerate(probs) if p > threshold])
    
    # 🔥 Add rule-based boost
    labels = set(keyword_boost(text, labels))
    
    if len(labels) == 0:
        return ["Other"]
    
    return list(labels)

In [470]:
def keyword_boost(text, labels):
    text = text.lower()
    
    if "water" in text or "paani" in text or "pipe" in text:
        labels.add("Water")
        
    if "bijli" in text or "electric" in text or "current" in text:
        labels.add("Electricity")
    
    return list(labels)

In [471]:
predict_complaint("paani leakage se short circuit ho raha hai")

['Electricity', 'Water']

In [472]:
predict_complaint("internet slow hai aur network problem aa rahi hai")

['Internet Services']

In [473]:
predict_complaint("road pe bahut gaddhe hain aur light bhi nahi aa rahi")

['Roads', 'Electricity']

In [474]:
predict_complaint("there is water coming from pipe and due to that bijli problem issue is happening")

['Electricity', 'Water']

In [475]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=mlb.classes_))

                   precision    recall  f1-score   support

          Banking       1.00      1.00      1.00       409
        Documents       1.00      1.00      1.00       406
        Education       1.00      1.00      1.00       403
      Electricity       1.00      1.00      1.00       400
       Employment       1.00      1.00      1.00       465
              Gas       1.00      1.00      1.00       415
       Healthcare       1.00      1.00      1.00       450
          Housing       1.00      1.00      1.00       431
        Insurance       1.00      1.00      1.00       428
Internet Services       1.00      1.00      1.00       409
      Law & Order       1.00      1.00      1.00       369
          Pension       1.00      1.00      1.00       416
         Railways       1.00      1.00      1.00       449
            Roads       1.00      1.00      1.00       432
       Sanitation       1.00      1.00      1.00       424
              Tax       1.00      1.00      1.00       

In [476]:
real_test_samples = [
    ("water coming from pipe and electricity problem is happening", ["Water", "Electricity"]),
    ("bijli issue hai aur light baar baar ja rahi hai", ["Electricity"]),
    ("internet ka speed bahut slow hai and network bhi nahi aa raha", ["Internet Services", "Telecom"]),
    ("road pe itne gaddhe hain ki chalna mushkil hai", ["Roads"]),
    ("kooda bahut jama ho gaya hai safai nahi ho rahi", ["Sanitation"]),
    ("pension 2 mahine se nahi aayi", ["Pension"]),
    ("hospital mein doctor available nahi hai", ["Healthcare"]),
    ("atm se paise kat gaye but transaction failed", ["Banking"]),
    ("gas cylinder leak ho raha hai", ["Gas"]),
    ("train late hai aur railway service kharab hai", ["Railways"]),
]

In [477]:
correct = 0

for text, actual in real_test_samples:
    pred = predict_complaint(text)
    
    print("\nTEXT:", text)
    print("ACTUAL:", actual)
    print("PREDICTED:", pred)
    
    if set(pred) == set(actual):
        correct += 1

print("\nReal Accuracy:", correct / len(real_test_samples))


TEXT: water coming from pipe and electricity problem is happening
ACTUAL: ['Water', 'Electricity']
PREDICTED: ['Electricity', 'Water']

TEXT: bijli issue hai aur light baar baar ja rahi hai
ACTUAL: ['Electricity']
PREDICTED: ['Electricity']

TEXT: internet ka speed bahut slow hai and network bhi nahi aa raha
ACTUAL: ['Internet Services', 'Telecom']
PREDICTED: ['Internet Services', 'Telecom']

TEXT: road pe itne gaddhe hain ki chalna mushkil hai
ACTUAL: ['Roads']
PREDICTED: ['Roads']

TEXT: kooda bahut jama ho gaya hai safai nahi ho rahi
ACTUAL: ['Sanitation']
PREDICTED: ['Sanitation']

TEXT: pension 2 mahine se nahi aayi
ACTUAL: ['Pension']
PREDICTED: ['Pension']

TEXT: hospital mein doctor available nahi hai
ACTUAL: ['Healthcare']
PREDICTED: ['Healthcare']

TEXT: atm se paise kat gaye but transaction failed
ACTUAL: ['Banking']
PREDICTED: ['Banking']

TEXT: gas cylinder leak ho raha hai
ACTUAL: ['Gas']
PREDICTED: ['Gas']

TEXT: train late hai aur railway service kharab hai
ACTUAL: ['R

In [478]:
print("Exact Match Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score (Micro):", f1_score(y_test, y_pred, average='micro'))

Exact Match Accuracy: 0.9988
F1 Score (Micro): 0.9996229260935143


In [479]:
hard_test_samples = [
    ("there is fluid leaking from the wall and causing electrical issues", ["Water", "Electricity"]),
    ("lights keep going off and voltage is unstable", ["Electricity"]),
    ("the streets are full of waste and smell is unbearable", ["Sanitation"]),
    ("my connection is extremely slow and keeps disconnecting", ["Internet Services"]),
    ("the road condition is very bad and full of pits", ["Roads"]),
    ("money deducted but not credited to account", ["Banking"]),
]

correct = 0

for text, actual in hard_test_samples:
    pred = predict_complaint(text)
    
    print("\nTEXT:", text)
    print("ACTUAL:", actual)
    print("PREDICTED:", pred)
    
    if set(pred) == set(actual):
        correct += 1

print("\nHard Test Accuracy:", correct / len(hard_test_samples))


TEXT: there is fluid leaking from the wall and causing electrical issues
ACTUAL: ['Water', 'Electricity']
PREDICTED: ['Housing', 'Electricity']

TEXT: lights keep going off and voltage is unstable
ACTUAL: ['Electricity']
PREDICTED: ['Electricity']

TEXT: the streets are full of waste and smell is unbearable
ACTUAL: ['Sanitation']
PREDICTED: ['Sanitation']

TEXT: my connection is extremely slow and keeps disconnecting
ACTUAL: ['Internet Services']
PREDICTED: ['Internet Services']

TEXT: the road condition is very bad and full of pits
ACTUAL: ['Roads']
PREDICTED: ['Roads']

TEXT: money deducted but not credited to account
ACTUAL: ['Banking']
PREDICTED: ['Banking']

Hard Test Accuracy: 0.8333333333333334


In [480]:
df_kaggle["clean_text"] = df_kaggle["Ticket Description"].apply(clean_text)

In [481]:
def map_category(text):
    text = text.lower()

    if any(w in text for w in ["payment", "transaction", "money", "refund", "account", "bank"]):
        return ["Banking"]

    elif any(w in text for w in ["internet", "wifi", "network", "slow", "connection"]):
        return ["Internet Services"]

    elif any(w in text for w in ["electricity", "power", "voltage", "bill", "current"]):
        return ["Electricity"]

    elif any(w in text for w in ["water", "leak", "pipe", "supply"]):
        return ["Water"]

    elif any(w in text for w in ["road", "pothole", "traffic"]):
        return ["Roads"]

    elif any(w in text for w in ["garbage", "waste", "dirty", "cleaning"]):
        return ["Sanitation"]

    elif any(w in text for w in ["house", "building", "wall", "construction"]):
        return ["Housing"]

    elif any(w in text for w in ["hospital", "doctor", "medical"]):
        return ["Healthcare"]

    elif any(w in text for w in ["train", "railway"]):
        return ["Railways"]

    elif any(w in text for w in ["gas", "cylinder"]):
        return ["Gas"]

    elif any(w in text for w in ["job", "salary", "employment"]):
        return ["Employment"]

    elif any(w in text for w in ["police", "theft", "crime"]):
        return ["Law & Order"]

    elif any(w in text for w in ["school", "college", "education"]):
        return ["Education"]

    elif any(w in text for w in ["document", "aadhaar", "pan"]):
        return ["Documents"]

    elif any(w in text for w in ["tax", "gst"]):
        return ["Tax"]

    else:
        return ["Other"]

In [482]:
df_kaggle["labels"] = df_kaggle["Ticket Description"].apply(map_category)

In [483]:
y_kaggle = mlb.transform(df_kaggle["labels"])

C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_label.py:900: UserWarning: unknown class(es) ['Other'] will be ignored
  warnings.warn(


In [484]:
X_kaggle = vectorizer.transform(df_kaggle["clean_text"])

In [485]:
y_probs = model.predict_proba(X_kaggle)
y_pred = (y_probs > 0.25).astype(int)

In [486]:
from sklearn.metrics import accuracy_score, f1_score

print("Kaggle Exact Accuracy:", accuracy_score(y_kaggle, y_pred))
print("Kaggle F1 Score (Micro):", f1_score(y_kaggle, y_pred, average='micro'))

Kaggle Exact Accuracy: 0.7545164718384697
Kaggle F1 Score (Micro): 0.5182394924662966


In [487]:
predict_complaint("water coming from pipe and electricity problem ho rahi hai")

['Electricity', 'Water']

In [488]:
correct = 0
total = len(hard_test_samples)

for text, actual in hard_test_samples:
    pred = predict_complaint(text)
    
    if set(pred) == set(actual):
        correct += 1

accuracy = correct / total   # ✅ DEFINE HERE

print("Hard Test Accuracy:", accuracy)

Hard Test Accuracy: 0.8333333333333334


In [489]:
from sklearn.metrics import f1_score
import numpy as np

y_true = []
y_pred = []

for text, actual in hard_test_samples:
    pred = predict_complaint(text)
    
    y_true.append(mlb.transform([actual])[0])
    y_pred.append(mlb.transform([pred])[0])

f1_micro = f1_score(np.array(y_true), np.array(y_pred), average='micro')

print("F1 Score:", f1_micro)

F1 Score: 0.8571428571428571


In [499]:
predict_complaint("no garbage collected and no paani from 3 days")

['Sanitation', 'Water']